In [9]:
# Vérification GPU disponible
import torch
print(f"GPU disponible : {torch.cuda.is_available()}")
print(f"Device : {torch.cuda.get_device_name(0)}")

# Installation des dépendances
!pip install monai nibabel dicom2nifti -q

GPU disponible : True
Device : Tesla T4


In [10]:
from google.colab import drive
drive.mount('/content/drive')

# Vérification des fichiers
import os

data_dir = '/content/drive/MyDrive/Liver_Segmentation/preprocessed'
model_dir = '/content/drive/MyDrive/Liver_Segmentation/results'

# Créer le dossier results s'il n'existe pas
os.makedirs(model_dir, exist_ok=True)

# Vérifier les fichiers train
train_images = os.listdir(os.path.join(data_dir, 'Train/images'))
train_labels = os.listdir(os.path.join(data_dir, 'Train/labels'))
test_images  = os.listdir(os.path.join(data_dir, 'Test/images'))

print(f"Train images : {len(train_images)}")
print(f"Train labels : {len(train_labels)}")
print(f"Test images  : {len(test_images)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Train images : 10
Train labels : 10
Test images  : 2


In [11]:
import os
import numpy as np
from glob import glob
import torch
from monai.utils import set_determinism
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, Spacingd, Orientationd,
    ScaleIntensityRanged, CropForegroundd, Resized, ToTensord,
    RandFlipd, RandRotate90d, RandGaussianNoised
)
from monai.data import Dataset, DataLoader, CacheDataset
from monai.networks.nets import UNet
from monai.networks.layers import Norm
from monai.losses import DiceLoss

# ── Paramètres ────────────────────────────────────────────────
device     = torch.device("cuda")  # GPU T4 ✅
data_dir   = '/content/drive/MyDrive/Liver_Segmentation/preprocessed'
model_dir  = '/content/drive/MyDrive/Liver_Segmentation/results'
pixdim     = (1.5, 1.5, 1.0)
a_min, a_max = -200, 200
spatial_size = [128, 128, 80]

# ── Fichiers ──────────────────────────────────────────────────
path_train_images = sorted(glob(os.path.join(data_dir, 'Train/images/*.nii')))
path_train_labels = sorted(glob(os.path.join(data_dir, 'Train/labels/*.nii')))
path_test_images  = sorted(glob(os.path.join(data_dir, 'Test/images/*.nii')))
path_test_labels  = sorted(glob(os.path.join(data_dir, 'Test/labels/*.nii')))

train_files = [{"image": img, "label": lbl} for img, lbl in zip(path_train_images, path_train_labels)]
test_files  = [{"image": img, "label": lbl} for img, lbl in zip(path_test_images,  path_test_labels)]

print(f"Train : {len(train_files)} volumes")
print(f"Test  : {len(test_files)} volumes")

# ── Transforms ────────────────────────────────────────────────
train_transforms = Compose([
    LoadImaged(keys=['image', 'label']),
    EnsureChannelFirstd(keys=['image', 'label']),
    Spacingd(keys=['image', 'label'], pixdim=pixdim, mode=("bilinear", "nearest")),
    Orientationd(keys=['image', 'label'], axcodes="RAS"),
    ScaleIntensityRanged(keys=['image'], a_min=a_min, a_max=a_max, b_min=0.0, b_max=1.0, clip=True),
    CropForegroundd(keys=['image', 'label'], source_key='image'),
    Resized(keys=['image', 'label'], spatial_size=spatial_size),
    RandFlipd(keys=['image', 'label'], prob=0.5, spatial_axis=0),
    RandFlipd(keys=['image', 'label'], prob=0.5, spatial_axis=1),
    RandFlipd(keys=['image', 'label'], prob=0.5, spatial_axis=2),
    RandRotate90d(keys=['image', 'label'], prob=0.5, max_k=3),
    RandGaussianNoised(keys=['image'], prob=0.3, mean=0.0, std=0.1),
    ToTensord(keys=['image', 'label']),
])

test_transforms = Compose([
    LoadImaged(keys=['image', 'label']),
    EnsureChannelFirstd(keys=['image', 'label']),
    Spacingd(keys=['image', 'label'], pixdim=pixdim, mode=("bilinear", "nearest")),
    Orientationd(keys=['image', 'label'], axcodes="RAS"),
    ScaleIntensityRanged(keys=['image'], a_min=a_min, a_max=a_max, b_min=0.0, b_max=1.0, clip=True),
    CropForegroundd(keys=['image', 'label'], source_key='image'),
    Resized(keys=['image', 'label'], spatial_size=spatial_size),
    ToTensord(keys=['image', 'label']),
])

# ── DataLoaders ───────────────────────────────────────────────
# CacheDataset sur GPU → charge tout en mémoire → beaucoup plus rapide
train_ds = CacheDataset(data=train_files, transform=train_transforms, cache_rate=1.0)
test_ds  = CacheDataset(data=test_files,  transform=test_transforms,  cache_rate=1.0)

train_loader = DataLoader(train_ds, batch_size=2, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=1, shuffle=False, num_workers=2)

print("✅ Pipeline prêt !")

Train : 10 volumes
Test  : 2 volumes


Loading dataset: 100%|██████████| 2/2 [00:03<00:00,  1.76s/it]

✅ Pipeline prêt !


In [12]:
# ── Modèle ────────────────────────────────────────────────────
model = UNet(
    spatial_dims=3,
    in_channels=1,
    out_channels=2,
    channels=(16, 32, 64, 128, 256),
    strides=(2, 2, 2, 2),
    num_res_units=2,
    norm=Norm.BATCH,
).to(device)

# ── Loss + Optimizer + Scheduler ──────────────────────────────
loss_function = DiceLoss(to_onehot_y=True, sigmoid=True, squared_pred=True)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4,
    amsgrad=True
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    factor=0.5,
    patience=15
)

print(f"✅ Modèle sur : {next(model.parameters()).device}")
print(f"Paramètres   : {sum(p.numel() for p in model.parameters()):,}")

✅ Modèle sur : cuda:0
Paramètres   : 4,808,917


In [13]:
from monai.losses import DiceLoss

def dice_metric(predicted, target):
    dice_value = DiceLoss(to_onehot_y=True, sigmoid=True, squared_pred=True)
    return 1 - dice_value(predicted, target).item()

# ── Paramètres d'entraînement ─────────────────────────────────
max_epochs    = 200
best_metric   = -1
best_metric_epoch = -1
save_loss_train, save_loss_test     = [], []
save_metric_train, save_metric_test = [], []

# ── Boucle ────────────────────────────────────────────────────
for epoch in range(max_epochs):
    print(f"{'─'*10} epoch {epoch+1}/{max_epochs}")
    model.train()
    train_epoch_loss, train_step, epoch_metric_train = 0, 0, 0

    for batch_data in train_loader:
        train_step += 1
        volume = batch_data["image"].to(device)
        label  = batch_data["label"].to(device)
        label  = label != 0

        optimizer.zero_grad()
        outputs    = model(volume)
        train_loss = loss_function(outputs, label)
        train_loss.backward()
        optimizer.step()

        train_epoch_loss   += train_loss.item()
        epoch_metric_train += dice_metric(outputs, label)

        print(f"  {train_step}/{len(train_loader)}, loss: {train_loss.item():.4f}, dice: {dice_metric(outputs, label):.4f}")

    train_epoch_loss   /= train_step
    epoch_metric_train /= train_step
    save_loss_train.append(train_epoch_loss)
    save_metric_train.append(epoch_metric_train)
    np.save(os.path.join(model_dir, 'loss_train.npy'),   save_loss_train)
    np.save(os.path.join(model_dir, 'metric_train.npy'), save_metric_train)
    print(f"  → Epoch loss: {train_epoch_loss:.4f} | Epoch dice: {epoch_metric_train:.4f}")

    # ── Évaluation ────────────────────────────────────────────
    model.eval()
    with torch.no_grad():
        test_epoch_loss, epoch_metric_test, test_step = 0, 0, 0

        for test_data in test_loader:
            test_step += 1
            test_volume = test_data["image"].to(device)
            test_label  = test_data["label"].to(device)
            test_label  = test_label != 0

            test_outputs = model(test_volume)
            test_loss    = loss_function(test_outputs, test_label)
            test_epoch_loss   += test_loss.item()
            epoch_metric_test += dice_metric(test_outputs, test_label)

        test_epoch_loss   /= test_step
        epoch_metric_test /= test_step
        save_loss_test.append(test_epoch_loss)
        save_metric_test.append(epoch_metric_test)
        np.save(os.path.join(model_dir, 'loss_test.npy'),   save_loss_test)
        np.save(os.path.join(model_dir, 'metric_test.npy'), save_metric_test)

        # Scheduler
        scheduler.step(epoch_metric_test)
        current_lr = optimizer.param_groups[0]['lr']
        print(f"  → Test loss: {test_epoch_loss:.4f} | Test dice: {epoch_metric_test:.4f} | lr: {current_lr:.2e}")

        if epoch_metric_test > best_metric:
            best_metric       = epoch_metric_test
            best_metric_epoch = epoch + 1
            torch.save(model.state_dict(), os.path.join(model_dir, 'best_metric_model.pth'))
            print(f"  ✅ Nouveau meilleur modèle sauvegardé ! Dice: {best_metric:.4f}")

        print(f"  Best dice: {best_metric:.4f} at epoch {best_metric_epoch}")

print(f"\n🏆 Entraînement terminé — best dice: {best_metric:.4f} at epoch {best_metric_epoch}")

────────── epoch 1/200
  1/5, loss: 0.5079, dice: 0.4921
  2/5, loss: 0.5322, dice: 0.4678
  3/5, loss: 0.4302, dice: 0.5698
  4/5, loss: 0.5401, dice: 0.4599
  5/5, loss: 0.4731, dice: 0.5269
  → Epoch loss: 0.4967 | Epoch dice: 0.5033
  → Test loss: 0.4237 | Test dice: 0.5763 | lr: 1.00e-04
  ✅ Nouveau meilleur modèle sauvegardé ! Dice: 0.5763
  Best dice: 0.5763 at epoch 1
────────── epoch 2/200
  1/5, loss: 0.5048, dice: 0.4952
  2/5, loss: 0.4672, dice: 0.5328
  3/5, loss: 0.5076, dice: 0.4924
  4/5, loss: 0.5168, dice: 0.4832
  5/5, loss: 0.4376, dice: 0.5624
  → Epoch loss: 0.4868 | Epoch dice: 0.5132
  → Test loss: 0.4209 | Test dice: 0.5791 | lr: 1.00e-04
  ✅ Nouveau meilleur modèle sauvegardé ! Dice: 0.5791
  Best dice: 0.5791 at epoch 2
────────── epoch 3/200
  1/5, loss: 0.4349, dice: 0.5651
  2/5, loss: 0.5296, dice: 0.4704
  3/5, loss: 0.3942, dice: 0.6058
  4/5, loss: 0.5156, dice: 0.4844
  5/5, loss: 0.5201, dice: 0.4799
  → Epoch loss: 0.4789 | Epoch dice: 0.5211
  → T